<center>
<img src="https://www.infnet.edu.br/infnet/wp-content/uploads/sites/18/2021/10/infnet-30-horizontal-padrao@300x-8-1024x265.png" width="60%"/>
</center>

# MBA em Engenharia de Dados: Big Data e IA
## Processamento de Big Data com Apache Spark e Spark SQL [26E3_2]
### Projeto da disciplina

### Variáveis dos volumes paths

In [0]:
CATALOG                     = 'instacart'

RAW_AISLES                  = f'/Volumes/{CATALOG}/raw/aisle/'
RAW_DEPARTMENTS             = f'/Volumes/{CATALOG}/raw/department/'
RAW_ORDERS                  = f'/Volumes/{CATALOG}/raw/order/'
RAW_PRODUCTS                = f'/Volumes/{CATALOG}/raw/product/'
RAW_ORDER_PRODUCTS_PRIOR    = f'/Volumes/{CATALOG}/raw/order_product_prior/'
RAW_ORDER_PRODUCTS_TRAIN    = f'/Volumes/{CATALOG}/raw/order_product_train/'

BRONZE_AISLES               = f'{CATALOG}.bronze.aisle'
BRONZE_DEPARTMENTS          = f'{CATALOG}.bronze.department'
BRONZE_ORDERS               = f'{CATALOG}.bronze.order'
BRONZE_PRODUCTS             = f'{CATALOG}.bronze.product'
BRONZE_ORDER_PRODUCTS_PRIOR = f'{CATALOG}.bronze.order_product_prior'
BRONZE_ORDER_PRODUCTS_TRAIN = f'{CATALOG}.bronze.order_product_train'

## CAMADA BRONZE

### Leitura e exploração dos dados

In [0]:
aisles_df = spark.read.csv(RAW_AISLES, header=True)
departments_df = spark.read.csv(RAW_DEPARTMENTS, header=True)
orders_df = spark.read.csv(RAW_ORDERS, header=True)
products_df = spark.read.csv(RAW_PRODUCTS, header=True, quote='"', escape='"', multiLine=True,)
order_products_prior_df = spark.read.csv(RAW_ORDER_PRODUCTS_PRIOR, header=True)
order_products_train_df = spark.read.csv(RAW_ORDER_PRODUCTS_TRAIN, header=True) 

In [0]:
print(f"Quantidade de registros em aisles: {aisles_df.count()}")
print(f"Quantidade de registros em departments: {departments_df.count()}")
print(f"Quantidade de registros em orders: {orders_df.count()}")
print(f"Quantidade de registros em products: {products_df.count()}")
print(f"Quantidade de registros em products_order_prior: {order_products_prior_df.count()}")
print(f"Quantidade de registros em products_order_train: {order_products_train_df.count()}")

In [0]:
aisles_df.printSchema()

In [0]:
aisles_df.display()

In [0]:
departments_df.printSchema()

In [0]:
departments_df.display()

In [0]:
orders_df.printSchema()

In [0]:
orders_df.display()

In [0]:
products_df.printSchema()

In [0]:
products_df.display()

In [0]:
order_products_prior_df.printSchema()

In [0]:
order_products_prior_df.display()

In [0]:
order_products_train_df.printSchema()

In [0]:
order_products_train_df.display()

### Convertendo dados para o formato delta e salvando na camada bronze

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS instacart.bronze;

CREATE TABLE IF NOT EXISTS instacart.bronze.aisle (  
    aisle_id STRING,
    aisle STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.bronze.department (  
    department_id STRING,
    department STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.bronze.order (  
    order_id STRING,
    user_id STRING,
    eval_set STRING,
    order_number STRING,
    order_dow STRING,
    order_hour_of_day  STRING,
    days_since_prior_order STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.bronze.product (  
    product_id STRING,
    product_name STRING,
    aisle_id STRING,
    department_id STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.bronze.order_product_prior (  
    order_id STRING,
    product_id STRING,
    add_to_cart_order STRING,
    reordered STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.bronze.order_product_train (  
    order_id STRING,
    product_id STRING,
    add_to_cart_order STRING,
    reordered STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
# A função abaixo é genérica independente da camada,
# já que isso é controlado pela variável "layer_path".
# Além disso, "merge_keys" controla justamente
# quais as colunas serão utilizadas para fazer o merge dos dados

from delta.tables import DeltaTable

def merge_data(df, layer_path, merge_keys):
    delta = DeltaTable.forName(spark, layer_path)

    (delta.alias("old")
    .merge(
        df.alias("new"),
        " AND ".join(f"(new.{col} = old.{col})" for col in merge_keys),
    )
    .whenNotMatchedInsertAll()
    .execute()
    )

In [0]:
merge_data(df=aisles_df, layer_path=BRONZE_AISLES, merge_keys=['aisle_id'])
merge_data(df=departments_df, layer_path=BRONZE_DEPARTMENTS, merge_keys=['department_id'])
merge_data(df=orders_df, layer_path=BRONZE_ORDERS, merge_keys=['order_id'])
merge_data(df=products_df, layer_path=BRONZE_PRODUCTS, merge_keys=['product_id'])
merge_data(df=order_products_train_df, layer_path=BRONZE_ORDER_PRODUCTS_TRAIN, merge_keys=order_products_train_df.columns)
merge_data(df=order_products_prior_df, layer_path=BRONZE_ORDER_PRODUCTS_PRIOR, merge_keys=order_products_train_df.columns)